# STATE K562 Perturbation Benchmark

Runs Arc Institute's **ST-Parse** model on the Replogle K562 Perturb-seq dataset.
Computes per-gene Pearson correlations for integration with GLMP class analysis.

**Prerequisites:** Set runtime to **T4 GPU** (Runtime > Change runtime type > T4)

**Estimated time:** 1-3 hours total

## Cell 1: Install dependencies

In [ ]:
!pip install -q arc-state scanpy anndata scipy huggingface_hub

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU ready: {gpu} ({mem:.1f} GB)")
else:
    print("WARNING: No GPU! Set Runtime > Change runtime type > T4 GPU")

## Cell 2: Download K562 data and ST-Parse model

Downloads ~1.5 GB of data and the model weights. Takes 5-10 minutes.

In [ ]:
import os, urllib.request
from huggingface_hub import snapshot_download

DATA_PATH = "ReplogleWeissman2022_K562_essential.h5ad"
MODEL_DIR = "ST-Parse"

# Download K562 Perturb-seq data
if not os.path.exists(DATA_PATH):
    print("Downloading K562 data (1.5 GB)...")
    url = "https://zenodo.org/records/7041849/files/ReplogleWeissman2022_K562_essential.h5ad?download=1"
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"Downloaded: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")
else:
    print(f"Data already present: {os.path.getsize(DATA_PATH)/1e9:.1f} GB")

# Download ST-Parse model
if not os.path.exists(MODEL_DIR):
    print("Downloading ST-Parse model from HuggingFace...")
    snapshot_download("arcinstitute/ST-Parse", local_dir=MODEL_DIR)
    print("Model downloaded.")
else:
    print("Model already present.")

print("\nModel files:")
for root, dirs, files in os.walk(MODEL_DIR):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path} ({os.path.getsize(path)/1e6:.0f} MB)")

## Cell 3: Load and preprocess data

In [ ]:
import scanpy as sc
import numpy as np

print("Loading K562 data...")
adata = sc.read_h5ad(DATA_PATH)
print(f"Shape: {adata.shape}")
print(f"Obs columns: {list(adata.obs.columns)}")

# Identify perturbation column and control label
pert_col = None
for col in ["gene", "perturbation", "guide_id", "condition", "target_gene"]:
    if col in adata.obs.columns:
        pert_col = col
        break

ctrl_label = None
for label in ["non-targeting", "control", "ctrl", "non_targeting"]:
    if label in adata.obs[pert_col].values:
        ctrl_label = label
        break

perts = [p for p in adata.obs[pert_col].unique() if p != ctrl_label]
print(f"Perturbation column: '{pert_col}'")
print(f"Control label: '{ctrl_label}'")
print(f"Perturbation targets: {len(perts)}")
print(f"Control cells: {(adata.obs[pert_col] == ctrl_label).sum()}")

## Cell 4: Preprocess for STATE inference

In [ ]:
PROC_PATH = "k562_for_state.h5ad"

print("Preprocessing...")
adata_proc = adata.copy()
sc.pp.normalize_total(adata_proc, target_sum=1e4)
sc.pp.log1p(adata_proc)
sc.pp.highly_variable_genes(adata_proc, n_top_genes=2000)

hvg_idx = np.where(adata_proc.var.highly_variable)[0]
X_hvg = adata_proc.X[:, hvg_idx]
if hasattr(X_hvg, 'toarray'):
    X_hvg = X_hvg.toarray()
adata_proc.obsm["X_hvg"] = X_hvg
print(f"HVG matrix: {X_hvg.shape}")

adata_proc.write(PROC_PATH)
print(f"Saved preprocessed data: {PROC_PATH}")

## Cell 5: Run STATE inference

This is the long step (30 min to 3 hours depending on GPU). You can watch progress below.

In [ ]:
import subprocess, time, os

OUTPUT_PATH = "state_predictions.h5ad"

# Find checkpoint
ckpt_path = None
for root, dirs, files in os.walk(MODEL_DIR):
    for f in sorted(files):
        if f.endswith((".ckpt", ".safetensors", ".pt", ".bin")):
            ckpt_path = os.path.join(root, f)
            break
    if ckpt_path:
        break

print(f"Checkpoint: {ckpt_path}")
print(f"Starting STATE inference...")

t0 = time.time()
cmd = [
    "state", "tx", "infer",
    "--model-dir", MODEL_DIR,
    "--adata", PROC_PATH,
    "--pert-col", pert_col,
    "--embed-key", "X_hvg",
    "--output", OUTPUT_PATH,
]
if ckpt_path:
    cmd.extend(["--checkpoint", ckpt_path])

print(f"Command: {' '.join(cmd)}\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

elapsed = (time.time() - t0) / 60
print(f"\nFinished in {elapsed:.1f} minutes (exit code: {proc.returncode})")

if proc.returncode != 0:
    print("\n--- INFERENCE FAILED ---")
    print("If the CLI doesn't work, try the alternative in Cell 5b below.")

## Cell 5b (ALTERNATIVE): Python API fallback

Only run this if Cell 5 failed. Attempts direct Python import.

In [ ]:
# Uncomment and run ONLY if Cell 5 failed:

# import importlib, sys
# try:
#     state_mod = importlib.import_module("state")
#     print(f"STATE module found: {dir(state_mod)}")
#     # Attempt direct inference call
#     from state.tx.infer import run_inference
#     run_inference(
#         model_dir=MODEL_DIR,
#         adata_path=PROC_PATH,
#         pert_col=pert_col,
#         embed_key="X_hvg",
#         output_path=OUTPUT_PATH,
#     )
# except Exception as e:
#     print(f"Error: {e}")
#     print("\nPlease check STATE docs for current API:")
#     print("https://github.com/ArcInstitute/state")

## Cell 6: Score predictions — per-gene Pearson correlation

In [ ]:
import scanpy as sc
import numpy as np
from scipy import stats
import csv

print("Loading predictions and observed data...")
adata_pred = sc.read_h5ad(OUTPUT_PATH)
adata_obs = sc.read_h5ad(DATA_PATH)

# Normalize observed data identically
sc.pp.normalize_total(adata_obs, target_sum=1e4)
sc.pp.log1p(adata_obs)

print(f"Predictions shape: {adata_pred.shape}")
print(f"Observed shape: {adata_obs.shape}")
print(f"Pred obs columns: {list(adata_pred.obs.columns[:10])}")
print(f"Pred layers: {list(adata_pred.layers.keys())}")
print(f"Pred obsm: {list(adata_pred.obsm.keys())}")

# Find shared gene space
shared = sorted(set(adata_pred.var_names) & set(adata_obs.var_names))
print(f"Shared genes: {len(shared)}")

if len(shared) == 0:
    print("\nNo shared var_names. STATE may use HVG indices.")
    print("Checking if prediction matrix matches HVG dimensions...")
    sc.pp.highly_variable_genes(adata_obs, n_top_genes=2000)
    hvg_names = adata_obs.var_names[adata_obs.var.highly_variable]
    if adata_pred.shape[1] == len(hvg_names):
        print(f"  Match! Using {len(hvg_names)} HVGs")
        adata_pred.var_names = hvg_names
        shared = list(hvg_names)
        adata_obs = adata_obs[:, hvg_names]

if len(shared) > 0:
    adata_pred = adata_pred[:, shared]
    adata_obs = adata_obs[:, shared]

In [ ]:
# Compute control mean
ctrl_mask = adata_obs.obs[pert_col] == ctrl_label
ctrl_X = adata_obs[ctrl_mask].X
if hasattr(ctrl_X, 'toarray'):
    ctrl_X = ctrl_X.toarray()
ctrl_mean = ctrl_X.mean(axis=0)

# Determine how predictions are structured
pred_pert_col = None
for col in [pert_col, "perturbation", "gene", "target_gene", "condition"]:
    if col in adata_pred.obs.columns:
        pred_pert_col = col
        break

print(f"Prediction perturbation column: {pred_pert_col}")
pred_perts = adata_pred.obs[pred_pert_col].unique() if pred_pert_col else []
print(f"Predicted perturbations: {len(pred_perts)}")

# Score each gene
scores = []
failed = 0

obs_perts = [p for p in adata_obs.obs[pert_col].unique() if p != ctrl_label]
print(f"\nScoring {len(obs_perts)} perturbations...")

for i, gene in enumerate(obs_perts):
    if (i+1) % 200 == 0:
        print(f"  [{i+1}/{len(obs_perts)}] scored={len(scores)}, failed={failed}")
    try:
        # Observed
        obs_mask = adata_obs.obs[pert_col] == gene
        n_obs = obs_mask.sum()
        if n_obs < 5:
            failed += 1
            continue
        obs_X = adata_obs[obs_mask].X
        if hasattr(obs_X, 'toarray'):
            obs_X = obs_X.toarray()
        obs_delta = np.array(obs_X.mean(axis=0) - ctrl_mean).flatten()

        # Predicted
        pred_mask = adata_pred.obs[pred_pert_col] == gene
        if pred_mask.sum() == 0:
            failed += 1
            continue
        pred_X = adata_pred[pred_mask].X
        if hasattr(pred_X, 'toarray'):
            pred_X = pred_X.toarray()
        pred_delta = np.array(pred_X.mean(axis=0) - ctrl_mean).flatten()

        # Pearson
        if np.std(obs_delta) > 1e-10 and np.std(pred_delta) > 1e-10:
            r, p = stats.pearsonr(obs_delta, pred_delta)
        else:
            r, p = 0.0, 1.0

        scores.append({
            "gene": gene,
            "pearson_correlation": round(float(r), 6),
            "pearson_pvalue": round(float(p), 10),
            "n_perturbed_cells": int(n_obs),
            "method": "STATE_ST-Parse",
        })
    except Exception as e:
        failed += 1

print(f"\nDone: {len(scores)} scored, {failed} failed")
if scores:
    accs = [s['pearson_correlation'] for s in scores]
    print(f"Mean Pearson r: {np.mean(accs):.4f}")
    print(f"Median: {np.median(accs):.4f}")

## Cell 7: Save results and download

In [ ]:
OUTPUT_TSV = "state_k562_per_gene_scores.tsv"

with open(OUTPUT_TSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "gene", "pearson_correlation", "pearson_pvalue",
        "n_perturbed_cells", "method"
    ], delimiter="\t")
    writer.writeheader()
    writer.writerows(scores)

print(f"Saved: {OUTPUT_TSV} ({len(scores)} genes)")

# Quick histogram
import matplotlib.pyplot as plt
accs = [s['pearson_correlation'] for s in scores]
plt.figure(figsize=(10, 4))
plt.hist(accs, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(accs), color='red', linestyle='--', label=f'Mean={np.mean(accs):.3f}')
plt.xlabel('Pearson r')
plt.ylabel('Count')
plt.title('STATE ST-Parse: Per-gene prediction accuracy on K562')
plt.legend()
plt.tight_layout()
plt.show()

## Cell 8: Per-class breakdown (upload gene_circuit_classes.tsv first)

Upload `gene_circuit_classes.tsv` using the file browser on the left, then run this cell.

In [ ]:
import csv
from collections import defaultdict

CLASSES_FILE = "gene_circuit_classes.tsv"

# Try to load classifications
try:
    classes = {}
    with open(CLASSES_FILE) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            classes[row["gene"]] = row["circuit_class"]
    print(f"Loaded {len(classes)} gene classifications")
except FileNotFoundError:
    print(f"Upload {CLASSES_FILE} first (file browser on left sidebar)")
    print("Or just download the TSV from Cell 7 and analyze locally.")
    classes = None

if classes:
    by_class = defaultdict(list)
    for s in scores:
        cls = classes.get(s["gene"], "unclassified")
        by_class[cls].append(s["pearson_correlation"])

    print("\n" + "="*60)
    print("STATE ST-Parse — Per GLMP Class Accuracy")
    print("="*60)
    for cls in ["I", "II", "III", "IV", "V", "unclassified"]:
        vals = by_class.get(cls, [])
        if vals:
            print(f"  Class {cls:5s}: N={len(vals):4d}, "
                  f"mean r={np.mean(vals):.4f}, "
                  f"median={np.median(vals):.4f}")

    # Class I vs III test
    c1 = by_class.get("I", [])
    c3 = by_class.get("III", [])
    if len(c3) >= 2:
        from scipy import stats
        u, p = stats.mannwhitneyu(c3, c1, alternative="less")
        print(f"\n  Class III vs I: Mann-Whitney (H1: III < I): p={p:.4f}")
        print(f"  Class I mean:   {np.mean(c1):.4f}")
        print(f"  Class III mean: {np.mean(c3):.4f}")

## Cell 9: Download results

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_TSV)
    print(f"Downloading {OUTPUT_TSV}...")
    print("\nAfter download, place in progframe/results/ and run:")
    print("  python merge_state_results.py")
except ImportError:
    print(f"Not in Colab. Results saved to: {OUTPUT_TSV}")